## Open In Colab

[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/decision_audit/cross_signal_decision_audit_colab.ipynb)


# Cross-Signal Decision Audit (Colab)

## Objective
Audit candidate-level uncertainty/risk signals as **decision objects** under thresholded control.

This notebook evaluates whether each signal is decision-grade for `failure_proxy_h15`, comparing:
- raw score
- Platt-calibrated score
- isotonic-calibrated score (when available)


## Hypotheses Tested

1. At least one planner-side proxy carries candidate-level discriminative signal.
2. Post-hoc calibration (Platt; isotonic when available) improves probability reliability metrics.
3. Thresholded decisions (`p <= tau`) can fail via false-safe, safe-reject, or feasible-set collapse.
4. Decision quality differs between nominal and shifted splits.


## Step 1 - Deterministic Bootstrap Constants


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

NOTEBOOK_NAME = 'cross_signal_decision_audit_colab'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))


## Step 2 - Storage Setup


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for p in (REPO_DIR, f'{REPO_DIR}/src'):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True. Re-run this notebook after runtime restart.')


## Step 4 - Configuration + Run Context


In [ ]:
from src.workflows import (
    initialize_risk_uq_run_context,
    load_experiment_config,
    report_risk_uq_run_context,
)

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

run_ctx = initialize_risk_uq_run_context(
    run_tag=RUN_TAG,
    run_tag_prefix=RUN_TAG_PREFIX,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    resume_mode=RUN_MODE,
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    planner_backend='latentdriver',
)

cfg = run_ctx.cfg
search_cfg = {}
RUN_TAG = str(run_ctx.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_ctx.shard_id)

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', cfg.run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
report_risk_uq_run_context(run_ctx, display_fn=display)

FOCUS_LABEL = str(run_cfg.get('focus_label', 'failure_proxy_h15'))
RISK_BUDGET_TAU = float(run_cfg.get('risk_budget_tau', 0.20))
TAU_SWEEP_VALUES = [round(float(x), 3) for x in list(__import__('numpy').linspace(0.05, 0.80, 16))]
BOOTSTRAP_SAMPLES = int(run_cfg.get('decision_bootstrap_samples', 300))
BOOTSTRAP_SEED = int(run_cfg.get('decision_bootstrap_seed', 17))
LOCAL_TAU_WINDOW = float(run_cfg.get('local_tau_window', 0.05))

cfg.uq_eval_probability_bins = int(max(10, int(getattr(cfg, 'uq_eval_probability_bins', 15))))


## Step 5 - Locate and Load Probe Artifacts


In [ ]:
import pandas as pd
from src.workflows import (
    discover_probe_run_prefixes,
    has_existing_miscalibration_probe_artifacts,
    load_existing_miscalibration_probe_bundle,
)

ANALYSIS_RUN_PREFIX_OVERRIDE = str(run_cfg.get('analysis_run_prefix', '')).strip()
MAX_DISCOVERED_RUNS = int(max(1, int(run_cfg.get('max_discovered_probe_runs', 50))))

analysis_run_prefix = ANALYSIS_RUN_PREFIX_OVERRIDE or str(cfg.run_prefix)
discovered_runs_df = discover_probe_run_prefixes(PERSIST_ROOT, limit=MAX_DISCOVERED_RUNS)

if not discovered_runs_df.empty:
    print('[artifacts] discovered probe runs (most recent first):')
    display(discovered_runs_df.head(20))
else:
    print('[artifacts] no probe runs discovered under PERSIST_ROOT:', PERSIST_ROOT)

# Strict mode: do NOT fall back to any previous run prefixes.
# analysis_run_prefix stays bound to current run context (or explicit override).

print('analysis_run_prefix =', analysis_run_prefix)
print('artifacts_present =', has_existing_miscalibration_probe_artifacts(analysis_run_prefix))

if bool(RUN_ENABLED) and (not has_existing_miscalibration_probe_artifacts(analysis_run_prefix)):
    raise FileNotFoundError(
        'No completed miscalibration probe artifacts were found. '
        'Run miscalibration_probe_colab.ipynb first, then re-run this notebook.'
    )

bundle = load_existing_miscalibration_probe_bundle(analysis_run_prefix)
pred_df = bundle.predictions_df.copy()
if pred_df.empty:
    raise RuntimeError('Probe bundle loaded but predictions frame is empty.')

print('pred_df rows =', len(pred_df), 'columns =', len(pred_df.columns))
print('focus_label_exists =', FOCUS_LABEL in pred_df.columns)
if FOCUS_LABEL not in pred_df.columns:
    raise ValueError(f'Focus label {FOCUS_LABEL!r} is missing from prediction artifacts.')

base_cols_preview = [
    c for c in [
        'scenario_id', 'step_idx', 'eval_split', 'shift_suite', FOCUS_LABEL,
        'planner_risk_top1_proxy', 'planner_risk_entropy_proxy', 'planner_risk_combo_proxy',
        'planner_risk_combo_platt', 'dist_std_max', 'belief_kl_current', 'min_ttc_h6', 'min_distance_h6',
        'progress_h6',
    ] if c in pred_df.columns
]
display(pred_df.loc[:, base_cols_preview].head(10))


## Step 6 - Build Signals and Calibration Variants

Signals audited:
- Required: `top1`, `entropy`, `combo`
- Additional baselines (added when available): `stdmax`, `belief_kl`, `inv_ttc`, `inv_distance`

Calibration variants:
- `raw`
- `platt`
- `iso` (isotonic; only if sklearn isotonic is available and calibration data is sufficient)


In [ ]:
import numpy as np
import pandas as pd

from src.risk_model.benchmark import (
    adaptive_ece,
    binary_auprc,
    binary_auroc,
    binary_ece,
    brier_score,
    nll_score,
)
from src.workflows.miscalibration_probe_flow import (
    _apply_binary_logistic_1d,
    _fit_binary_logistic_1d,
)

try:
    from sklearn.isotonic import IsotonicRegression
    HAS_ISOTONIC = True
except Exception:
    HAS_ISOTONIC = False
    IsotonicRegression = None

def _clip_prob(x):
    arr = np.asarray(pd.to_numeric(x, errors='coerce'), dtype=float)
    arr = np.where(np.isfinite(arr), arr, 0.5)
    return np.clip(arr, 1e-6, 1.0 - 1e-6)

def _safe_float(x, default=0.0):
    arr = np.asarray(pd.to_numeric(x, errors='coerce'), dtype=float)
    arr = np.where(np.isfinite(arr), arr, float(default))
    return arr

work_df = pred_df.copy()
if 'shift_suite' not in work_df.columns:
    work_df['shift_suite'] = 'nominal_clean'
else:
    work_df['shift_suite'] = work_df['shift_suite'].fillna('nominal_clean').astype(str)

if ('planner_risk_top1_proxy' not in work_df.columns) and ('dist_top1_weight' in work_df.columns):
    work_df['planner_risk_top1_proxy'] = 1.0 - _clip_prob(work_df['dist_top1_weight'])

if ('planner_risk_entropy_proxy' not in work_df.columns) and ('dist_entropy' in work_df.columns) and ('dist_num_components' in work_df.columns):
    entropy = _safe_float(work_df['dist_entropy'], default=0.0)
    n_comp = np.maximum(2.0, _safe_float(work_df['dist_num_components'], default=2.0))
    entropy_max = np.log(n_comp)
    entropy_max = np.where(np.isfinite(entropy_max) & (entropy_max > 1e-8), entropy_max, 1.0)
    conf_entropy = 1.0 - np.clip(entropy / entropy_max, 0.0, 1.0)
    work_df['planner_risk_entropy_proxy'] = 1.0 - conf_entropy

if ('planner_risk_combo_proxy' not in work_df.columns) and ('planner_risk_top1_proxy' in work_df.columns) and ('planner_risk_entropy_proxy' in work_df.columns):
    conf_top1 = 1.0 - _clip_prob(work_df['planner_risk_top1_proxy'])
    conf_entropy = 1.0 - _clip_prob(work_df['planner_risk_entropy_proxy'])
    combo_conf = np.clip(0.70 * conf_top1 + 0.30 * conf_entropy, 0.0, 1.0)
    work_df['planner_risk_combo_proxy'] = 1.0 - combo_conf

# Scenario-level split assumptions.
if 'eval_split' in work_df.columns:
    eval_split = work_df['eval_split'].astype(str)
    cal_mask = eval_split.eq('val')
    eval_mask = eval_split.isin(['test', 'high_interaction_holdout'])
    split_source = 'eval_split column from dataset artifacts'
else:
    # Fallback scenario split when eval_split is absent.
    if 'scenario_id' not in work_df.columns:
        work_df['scenario_id'] = np.arange(len(work_df)).astype(str)
    scenario_ids = pd.Series(work_df['scenario_id'].astype(str).unique())
    rng = np.random.default_rng(17)
    shuffled = scenario_ids.sample(frac=1.0, random_state=17).to_numpy(dtype=object)
    n = len(shuffled)
    n_val = max(1, int(round(0.15 * n)))
    n_test = max(1, int(round(0.15 * n)))
    val_ids = set(shuffled[:n_val].tolist())
    test_ids = set(shuffled[n_val:n_val + n_test].tolist())
    sid = work_df['scenario_id'].astype(str)
    cal_mask = sid.isin(val_ids)
    eval_mask = sid.isin(test_ids)
    split_source = 'fallback scenario-level random split (15% val, 15% eval)'

if int(cal_mask.sum()) < 50:
    print('[warning] calibration split has very few rows; using full dataset for calibrator fitting fallback.')
    cal_mask = pd.Series(np.ones((len(work_df),), dtype=bool), index=work_df.index)

if int(eval_mask.sum()) < 50:
    print('[warning] evaluation split has very few rows; using all rows for audit fallback.')
    eval_mask = pd.Series(np.ones((len(work_df),), dtype=bool), index=work_df.index)

print('split_source =', split_source)
print('calibration_rows =', int(cal_mask.sum()), 'evaluation_rows =', int(eval_mask.sum()))

def _fit_quantile_scaler(values, mask, q_lo=0.01, q_hi=0.99):
    arr = _safe_float(values, default=np.nan)
    ref = arr[np.asarray(mask, dtype=bool)]
    ref = ref[np.isfinite(ref)]
    if ref.size == 0:
        ref = arr[np.isfinite(arr)]
    if ref.size == 0:
        return (0.0, 1.0)
    lo = float(np.quantile(ref, q_lo))
    hi = float(np.quantile(ref, q_hi))
    if not np.isfinite(lo):
        lo = float(np.nanmin(ref))
    if not np.isfinite(hi):
        hi = float(np.nanmax(ref))
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi):
        hi = lo + 1.0
    if hi <= lo + 1e-9:
        hi = lo + 1.0
    return (lo, hi)

def _scale_quantile(values, lo, hi):
    arr = _safe_float(values, default=lo)
    out = (arr - float(lo)) / max(1e-9, float(hi - lo))
    return np.clip(out, 0.0, 1.0)

def _inverse_ttc(values, cap=8.0):
    ttc = _safe_float(values, default=cap)
    ttc = np.clip(ttc, 0.0, cap)
    return np.clip(1.0 - (ttc / max(1e-6, cap)), 0.0, 1.0)

def _inverse_distance(values, cap=20.0):
    d = _safe_float(values, default=cap)
    d = np.clip(d, 0.0, cap)
    return np.clip(1.0 - (d / max(1e-6, cap)), 0.0, 1.0)

signals = {}
if 'planner_risk_top1_proxy' in work_df.columns:
    signals['top1'] = _clip_prob(work_df['planner_risk_top1_proxy'])
if 'planner_risk_entropy_proxy' in work_df.columns:
    signals['entropy'] = _clip_prob(work_df['planner_risk_entropy_proxy'])
if 'planner_risk_combo_proxy' in work_df.columns:
    signals['combo'] = _clip_prob(work_df['planner_risk_combo_proxy'])

if 'dist_std_max' in work_df.columns:
    lo, hi = _fit_quantile_scaler(work_df['dist_std_max'], cal_mask)
    signals['stdmax'] = _clip_prob(_scale_quantile(work_df['dist_std_max'], lo, hi))
if 'belief_kl_current' in work_df.columns:
    lo, hi = _fit_quantile_scaler(work_df['belief_kl_current'], cal_mask)
    signals['belief_kl'] = _clip_prob(_scale_quantile(work_df['belief_kl_current'], lo, hi))
if 'min_ttc_h6' in work_df.columns:
    signals['inv_ttc'] = _clip_prob(_inverse_ttc(work_df['min_ttc_h6'], cap=8.0))
if 'min_distance_h6' in work_df.columns:
    signals['inv_distance'] = _clip_prob(_inverse_distance(work_df['min_distance_h6'], cap=20.0))

required = ['top1', 'entropy', 'combo']
missing_required = [x for x in required if x not in signals]
if missing_required:
    raise RuntimeError(f'Missing required baseline signals: {missing_required}.')

y_all = (_safe_float(work_df[FOCUS_LABEL], default=0.0) > 0.5).astype(float)
y_cal = y_all[np.asarray(cal_mask, dtype=bool)]

calibrator_rows = []
variant_to_column = {}
for name, risk_arr in signals.items():
    raw_col = f'risk_{name}_raw'
    platt_col = f'risk_{name}_platt'
    iso_col = f'risk_{name}_iso'

    work_df[raw_col] = _clip_prob(risk_arr)
    variant_to_column[f'{name}:raw'] = raw_col

    x_cal = _safe_float(work_df.loc[cal_mask, raw_col], default=0.5)
    y_cal = (_safe_float(work_df.loc[cal_mask, FOCUS_LABEL], default=0.0) > 0.5).astype(float)

    has_pos = bool(np.any(y_cal > 0.5))
    has_neg = bool(np.any(y_cal < 0.5))
    fit_ok = bool((len(y_cal) >= 30) and has_pos and has_neg)

    if fit_ok:
        alpha, beta = _fit_binary_logistic_1d(x_cal, y_cal)
        p_platt = _apply_binary_logistic_1d(_safe_float(work_df[raw_col], default=0.5), alpha, beta)
        work_df[platt_col] = _clip_prob(p_platt)
        variant_to_column[f'{name}:platt'] = platt_col
        calibrator_rows.append({
            'signal': name,
            'calibration': 'platt',
            'fitted': 1,
            'alpha': float(alpha),
            'beta': float(beta),
            'n_cal_rows': int(len(y_cal)),
            'pos_rate_cal': float(np.mean(y_cal)) if len(y_cal) else np.nan,
        })
    else:
        work_df[platt_col] = work_df[raw_col].to_numpy(dtype=float)
        variant_to_column[f'{name}:platt'] = platt_col
        calibrator_rows.append({
            'signal': name,
            'calibration': 'platt',
            'fitted': 0,
            'alpha': np.nan,
            'beta': np.nan,
            'n_cal_rows': int(len(y_cal)),
            'pos_rate_cal': float(np.mean(y_cal)) if len(y_cal) else np.nan,
        })

    iso_fitted = 0
    if HAS_ISOTONIC and fit_ok:
        x_iso = _safe_float(work_df.loc[cal_mask, raw_col], default=0.5)
        y_iso = (_safe_float(work_df.loc[cal_mask, FOCUS_LABEL], default=0.0) > 0.5).astype(float)
        unique_x = np.unique(np.round(x_iso, 6))
        if len(unique_x) >= 10:
            iso_model = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
            iso_model.fit(x_iso, y_iso)
            work_df[iso_col] = _clip_prob(iso_model.predict(_safe_float(work_df[raw_col], default=0.5)))
            variant_to_column[f'{name}:iso'] = iso_col
            iso_fitted = 1
        else:
            work_df[iso_col] = work_df[platt_col].to_numpy(dtype=float)
            variant_to_column[f'{name}:iso'] = iso_col
    else:
        work_df[iso_col] = work_df[platt_col].to_numpy(dtype=float)
        variant_to_column[f'{name}:iso'] = iso_col

    calibrator_rows.append({
        'signal': name,
        'calibration': 'iso',
        'fitted': int(iso_fitted),
        'alpha': np.nan,
        'beta': np.nan,
        'n_cal_rows': int(len(y_cal)),
        'pos_rate_cal': float(np.mean(y_cal)) if len(y_cal) else np.nan,
        'isotonic_available': int(HAS_ISOTONIC),
    })

calibrator_df = pd.DataFrame(calibrator_rows)
print('isotonic_available =', HAS_ISOTONIC)
print('signals =', sorted(signals.keys()))
print('variant_count =', len(variant_to_column))
display(calibrator_df)


## Step 7 - Metrics, Tau Sweep, CIs, and Sufficiency Gates

Metrics computed per `(shift_suite, signal, calibration)`:
- AUROC, PR-AUC, within-step AUC
- ECE, adaptive ECE, NLL, Brier
- local calibration around `tau` (window = `LOCAL_TAU_WINDOW`)

Decision metrics over `tau in [0.05, 0.80]`:
- accept_rate, false_safe, safe_reject
- feasible_set_rate, fallback_rate

All confidence intervals use scenario-level bootstrap.


In [ ]:
import numpy as np
import pandas as pd

eval_df = work_df.loc[eval_mask].copy()
if eval_df.empty:
    eval_df = work_df.copy()

if 'shift_suite' not in eval_df.columns:
    eval_df['shift_suite'] = 'nominal_clean'

if 'scenario_id' not in eval_df.columns:
    eval_df['scenario_id'] = np.arange(len(eval_df)).astype(str)

if 'step_idx' not in eval_df.columns:
    eval_df['step_idx'] = 0

eval_df['scenario_id'] = eval_df['scenario_id'].astype(str)
eval_df['step_idx'] = pd.to_numeric(eval_df['step_idx'], errors='coerce').fillna(0).astype(int)
eval_df['_step_key'] = eval_df['scenario_id'] + '::' + eval_df['step_idx'].astype(str)

y_col = FOCUS_LABEL

def _within_step_auc(df, prob_col, label_col):
    aucs = []
    for _, grp in df.groupby('_step_key', sort=False):
        y = (_safe_float(grp[label_col], default=0.0) > 0.5).astype(float)
        if (len(np.unique(y)) < 2) or (len(y) < 2):
            continue
        p = _clip_prob(grp[prob_col])
        auc = binary_auroc(p, y)
        if np.isfinite(auc):
            aucs.append(float(auc))
    if len(aucs) == 0:
        return np.nan, 0
    return float(np.mean(aucs)), int(len(aucs))

def _local_calibration(df, prob_col, label_col, tau, window):
    p = _clip_prob(df[prob_col])
    y = (_safe_float(df[label_col], default=0.0) > 0.5).astype(float)
    mask = np.abs(p - float(tau)) <= float(window)
    n = int(np.sum(mask))
    if n <= 0:
        return {'local_n': 0, 'local_mean_pred': np.nan, 'local_event_rate': np.nan, 'local_gap': np.nan}
    mean_pred = float(np.mean(p[mask]))
    event_rate = float(np.mean(y[mask]))
    return {
        'local_n': n,
        'local_mean_pred': mean_pred,
        'local_event_rate': event_rate,
        'local_gap': float(event_rate - mean_pred),
    }

def _decision_metrics(df, prob_col, label_col, tau):
    p = _clip_prob(df[prob_col])
    y = (_safe_float(df[label_col], default=0.0) > 0.5).astype(float)
    accepted = p <= float(tau)
    rejected = ~accepted
    accept_count = int(np.sum(accepted))
    reject_count = int(np.sum(rejected))
    false_safe = float(np.mean(y[accepted])) if accept_count > 0 else np.nan
    safe_reject = float(np.mean((y <= 0.5)[rejected])) if reject_count > 0 else np.nan
    feasible = np.nan
    fallback = np.nan
    if '_step_key' in df.columns:
        tmp = pd.DataFrame({'step_key': df['_step_key'].astype(str), 'accepted': accepted.astype(int)})
        feasible = float(tmp.groupby('step_key', sort=False)['accepted'].max().mean()) if len(tmp) > 0 else np.nan
        fallback = float(1.0 - feasible) if np.isfinite(feasible) else np.nan
    return {
        'n_rows': int(len(df)),
        'n_pos': int(np.sum(y > 0.5)),
        'accept_count': accept_count,
        'reject_count': reject_count,
        'accept_rate': float(np.mean(accepted)) if len(df) else np.nan,
        'false_safe': false_safe,
        'safe_reject': safe_reject,
        'feasible_set_rate': feasible,
        'fallback_rate': fallback,
    }

def _bootstrap_ci(metric_fn, df, n_boot=0, seed=17):
    n_boot = int(max(0, int(n_boot)))
    if (n_boot <= 0) or df.empty:
        return {}
    sid = df['scenario_id'].astype(str)
    groups = {k: v.index.to_numpy(dtype=int) for k, v in df.groupby(sid, sort=False)}
    keys = list(groups.keys())
    if len(keys) <= 1:
        return {}
    rng = np.random.default_rng(int(seed))
    samples = []
    for _ in range(n_boot):
        picked = rng.integers(0, len(keys), size=len(keys))
        idx = np.concatenate([groups[keys[int(i)]] for i in picked], axis=0)
        sub = df.iloc[idx]
        samples.append(metric_fn(sub))
    if len(samples) == 0:
        return {}
    keys0 = sorted(samples[0].keys())
    out = {}
    for k in keys0:
        arr = np.asarray([s.get(k, np.nan) for s in samples], dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size == 0:
            continue
        out[f'{k}_ci_low'] = float(np.quantile(arr, 0.025))
        out[f'{k}_ci_high'] = float(np.quantile(arr, 0.975))
    return out

metric_rows = []
tau_rows = []

variant_items = sorted(variant_to_column.items(), key=lambda kv: kv[0])
shifts = sorted(eval_df['shift_suite'].astype(str).unique().tolist())

for shift_suite in shifts:
    shift_df = eval_df[eval_df['shift_suite'].astype(str).eq(str(shift_suite))].copy()
    for variant, prob_col in variant_items:
        if prob_col not in shift_df.columns:
            continue
        signal_name, cal_name = variant.split(':', 1)

        def _global_metric_fn(sub_df):
            p = _clip_prob(sub_df[prob_col])
            y = (_safe_float(sub_df[y_col], default=0.0) > 0.5).astype(float)
            ws_auc, ws_n = _within_step_auc(sub_df, prob_col, y_col)
            local = _local_calibration(sub_df, prob_col, y_col, RISK_BUDGET_TAU, LOCAL_TAU_WINDOW)
            return {
                'auroc': float(binary_auroc(p, y)),
                'auprc': float(binary_auprc(p, y)),
                'within_step_auc': float(ws_auc),
                'ece': float(binary_ece(p, y, n_bins=int(cfg.uq_eval_probability_bins))),
                'adaptive_ece': float(adaptive_ece(p, y, n_bins=int(cfg.uq_eval_probability_bins))),
                'nll': float(nll_score(p, y)),
                'brier': float(brier_score(p, y)),
                'local_n': float(local['local_n']),
                'local_gap': float(local['local_gap']),
            }

        global_metrics = _global_metric_fn(shift_df)
        global_ci = _bootstrap_ci(_global_metric_fn, shift_df, n_boot=BOOTSTRAP_SAMPLES, seed=BOOTSTRAP_SEED)

        ws_auc, ws_n = _within_step_auc(shift_df, prob_col, y_col)
        local = _local_calibration(shift_df, prob_col, y_col, RISK_BUDGET_TAU, LOCAL_TAU_WINDOW)

        row = {
            'shift_suite': str(shift_suite),
            'domain': 'nominal' if str(shift_suite) == 'nominal_clean' else 'shifted',
            'signal': signal_name,
            'calibration': cal_name,
            'variant': variant,
            'prob_col': prob_col,
            'n_rows': int(len(shift_df)),
            'positive_rate': float(np.mean((_safe_float(shift_df[y_col], default=0.0) > 0.5).astype(float))),
            'within_step_groups_used': int(ws_n),
            'local_tau': float(RISK_BUDGET_TAU),
            'local_tau_window': float(LOCAL_TAU_WINDOW),
            **global_metrics,
            **global_ci,
        }
        metric_rows.append(row)

        for tau in TAU_SWEEP_VALUES:
            dec = _decision_metrics(shift_df, prob_col, y_col, tau=float(tau))

            def _tau_metric_fn(sub_df):
                return _decision_metrics(sub_df, prob_col, y_col, tau=float(tau))

            ci = _bootstrap_ci(_tau_metric_fn, shift_df, n_boot=BOOTSTRAP_SAMPLES, seed=BOOTSTRAP_SEED)

            gate_reasons = []
            if int(dec['n_rows']) < 200:
                gate_reasons.append('low_rows')
            if int(dec['n_pos']) < 20:
                gate_reasons.append('low_positives')
            if int(dec['accept_count']) < 30:
                gate_reasons.append('low_accept_count')
            if int(dec['reject_count']) < 30:
                gate_reasons.append('low_reject_count')
            if shift_df['_step_key'].nunique() < 30:
                gate_reasons.append('low_step_count')

            tau_rows.append({
                'shift_suite': str(shift_suite),
                'domain': 'nominal' if str(shift_suite) == 'nominal_clean' else 'shifted',
                'signal': signal_name,
                'calibration': cal_name,
                'variant': variant,
                'tau': float(tau),
                **dec,
                **ci,
                'status': 'ok' if len(gate_reasons) == 0 else 'inconclusive',
                'inconclusive_reason': ';'.join(gate_reasons),
            })

metrics_df = pd.DataFrame(metric_rows)
tau_sweep_df = pd.DataFrame(tau_rows)

if metrics_df.empty:
    raise RuntimeError('No metric rows were produced. Check signal availability and label column.')

# Domain-level comparison table (nominal vs shifted).
domain_compare_df = (
    metrics_df
    .groupby(['domain', 'signal', 'calibration'], as_index=False)
    .agg(
        n_rows=('n_rows', 'sum'),
        positive_rate=('positive_rate', 'mean'),
        auroc=('auroc', 'mean'),
        auprc=('auprc', 'mean'),
        within_step_auc=('within_step_auc', 'mean'),
        ece=('ece', 'mean'),
        nll=('nll', 'mean'),
        brier=('brier', 'mean'),
        local_gap=('local_gap', 'mean'),
        local_n=('local_n', 'sum'),
    )
    .sort_values(['domain', 'ece', 'nll'])
    .reset_index(drop=True)
)

tau_at_budget_df = pd.DataFrame()
if not tau_sweep_df.empty:
    tau_at_budget_df = (
        tau_sweep_df.assign(_dist=(tau_sweep_df['tau'] - float(RISK_BUDGET_TAU)).abs())
        .sort_values('_dist')
        .groupby(['shift_suite', 'signal', 'calibration'], as_index=False)
        .first()
        .drop(columns=['_dist'])
        .sort_values(['shift_suite', 'signal', 'calibration'])
        .reset_index(drop=True)
    )

print('metrics_df rows =', len(metrics_df))
print('tau_sweep_df rows =', len(tau_sweep_df))
print('inconclusive tau rows =', int((tau_sweep_df['status'] == 'inconclusive').sum()) if not tau_sweep_df.empty else 0)

display(metrics_df.head(30))
display(domain_compare_df.head(30))
if not tau_at_budget_df.empty:
    display(tau_at_budget_df.head(40))


## Step 8 - Paper-Ready Plots and Tables


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

figure_paths = {}

# Plot 1: Reliability curves (selected signals: top1/entropy/combo + one extra if available)
plot_signals = [s for s in ['top1', 'entropy', 'combo'] if s in set(metrics_df['signal'].astype(str))]
extras = [s for s in sorted(set(metrics_df['signal'].astype(str))) if s not in plot_signals]
if len(extras) > 0:
    plot_signals.append(extras[0])

plot_shifts = ['nominal_clean']
non_nom = [x for x in sorted(metrics_df['shift_suite'].astype(str).unique().tolist()) if x != 'nominal_clean']
if len(non_nom) > 0:
    plot_shifts.append(non_nom[0])

if len(plot_signals) > 0:
    n_rows = len(plot_shifts)
    n_cols = len(plot_signals)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.0 * n_cols, 4.2 * n_rows), sharex=True, sharey=True)
    if n_rows == 1 and n_cols == 1:
        axes = np.asarray([[axes]])
    elif n_rows == 1:
        axes = np.asarray([axes])
    elif n_cols == 1:
        axes = np.asarray([[a] for a in axes])

    rel_df = bundle.benchmark_bundle.reliability_df.copy()
    if rel_df.empty:
        rel_df = pd.DataFrame()

    for r, shift in enumerate(plot_shifts):
        for c, signal in enumerate(plot_signals):
            ax = axes[r, c]
            ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='ideal')
            for cal, color in [('raw', 'tab:red'), ('platt', 'tab:blue'), ('iso', 'tab:green')]:
                variant = f'{signal}:{cal}'
                col = variant_to_column.get(variant, '')
                if (not col) or (col not in eval_df.columns):
                    continue
                sub = eval_df[eval_df['shift_suite'].astype(str).eq(str(shift))].copy()
                if sub.empty:
                    continue
                p = _clip_prob(sub[col])
                y = (_safe_float(sub[FOCUS_LABEL], default=0.0) > 0.5).astype(float)
                bins = np.linspace(0.0, 1.0, int(cfg.uq_eval_probability_bins) + 1)
                xs = []
                ys = []
                for i in range(len(bins) - 1):
                    lo = bins[i]
                    hi = bins[i + 1]
                    if i == len(bins) - 2:
                        m = (p >= lo) & (p <= hi)
                    else:
                        m = (p >= lo) & (p < hi)
                    if not np.any(m):
                        continue
                    xs.append(float(np.mean(p[m])))
                    ys.append(float(np.mean(y[m])))
                if len(xs) > 0:
                    ax.plot(xs, ys, marker='o', linewidth=1.4, color=color, label=cal)
            ax.set_title(f'{signal} | {shift}')
            ax.set_xlabel('mean predicted probability')
            ax.set_ylabel('empirical failure rate')
            ax.grid(alpha=0.25)
            if r == 0 and c == 0:
                ax.legend(loc='best')
    fig.suptitle(f'Reliability audit ({FOCUS_LABEL})', y=1.01)
    fig.tight_layout()
    path = f'{analysis_run_prefix}_cross_signal_reliability.png'
    fig.savefig(path, dpi=170, bbox_inches='tight')
    figure_paths['cross_signal_reliability'] = path
    plt.show()

# Plot 2: Main metric comparison by signal/calibration
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['ece', 'nll', 'brier']):
    pivot = (
        domain_compare_df[domain_compare_df['domain'].astype(str).eq('nominal')]
        .pivot(index='signal', columns='calibration', values=metric)
        .sort_index()
    )
    if not pivot.empty:
        cols = [c for c in ['raw', 'platt', 'iso'] if c in pivot.columns]
        pivot = pivot.loc[:, cols]
        pivot.plot(kind='bar', ax=ax)
    ax.set_title(f'Nominal {metric.upper()}')
    ax.set_xlabel('signal')
    ax.set_ylabel(metric)
    ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
path = f'{analysis_run_prefix}_cross_signal_nominal_metrics.png'
fig.savefig(path, dpi=170, bbox_inches='tight')
figure_paths['cross_signal_nominal_metrics'] = path
plt.show()

# Plot 3: tau sweep for combo signal (or fallback first signal)
tau_plot_signal = 'combo' if 'combo' in set(tau_sweep_df['signal'].astype(str)) else str(tau_sweep_df['signal'].astype(str).iloc[0])
tau_plot_df = tau_sweep_df[tau_sweep_df['signal'].astype(str).eq(tau_plot_signal)].copy()
if not tau_plot_df.empty:
    shifts_for_plot = ['nominal_clean']
    nn = [x for x in sorted(tau_plot_df['shift_suite'].astype(str).unique().tolist()) if x != 'nominal_clean']
    if len(nn) > 0:
        shifts_for_plot.append(nn[0])

    metrics = ['accept_rate', 'false_safe', 'safe_reject', 'fallback_rate']
    fig, axes = plt.subplots(len(shifts_for_plot), len(metrics), figsize=(5.4 * len(metrics), 4.2 * len(shifts_for_plot)), sharex=True)
    if len(shifts_for_plot) == 1:
        axes = np.asarray([axes])

    for i, shift in enumerate(shifts_for_plot):
        sub_shift = tau_plot_df[tau_plot_df['shift_suite'].astype(str).eq(str(shift))]
        for j, metric in enumerate(metrics):
            ax = axes[i, j]
            for cal, color in [('raw', 'tab:red'), ('platt', 'tab:blue'), ('iso', 'tab:green')]:
                sub = sub_shift[sub_shift['calibration'].astype(str).eq(cal)].sort_values('tau')
                if sub.empty:
                    continue
                ax.plot(sub['tau'], sub[metric], marker='o', linewidth=1.5, color=color, label=cal)
                lo_col = f'{metric}_ci_low'
                hi_col = f'{metric}_ci_high'
                if lo_col in sub.columns and hi_col in sub.columns:
                    lo = pd.to_numeric(sub[lo_col], errors='coerce').to_numpy(dtype=float)
                    hi = pd.to_numeric(sub[hi_col], errors='coerce').to_numpy(dtype=float)
                    x = pd.to_numeric(sub['tau'], errors='coerce').to_numpy(dtype=float)
                    mask = np.isfinite(x) & np.isfinite(lo) & np.isfinite(hi)
                    if np.any(mask):
                        ax.fill_between(x[mask], lo[mask], hi[mask], color=color, alpha=0.14)
            if metric == 'false_safe':
                ax.axhline(float(RISK_BUDGET_TAU), color='k', linestyle='--', linewidth=1)
            ax.set_title(f'{shift} | {metric}')
            ax.set_xlabel('tau')
            ax.set_ylabel(metric)
            ax.grid(alpha=0.25)
            if i == 0 and j == 0:
                ax.legend(loc='best')
    fig.tight_layout()
    path = f'{analysis_run_prefix}_cross_signal_tau_sweep.png'
    fig.savefig(path, dpi=170, bbox_inches='tight')
    figure_paths['cross_signal_tau_sweep'] = path
    plt.show()

print('figure_paths =', figure_paths)


## Step 9 - Export Artifacts + Contract Manifest


In [ ]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest

metrics_path = f'{analysis_run_prefix}_cross_signal_metrics.csv'
tau_path = f'{analysis_run_prefix}_cross_signal_tau_sweep.csv'
domain_path = f'{analysis_run_prefix}_cross_signal_domain_compare.csv'
budget_path = f'{analysis_run_prefix}_cross_signal_tau_at_budget.csv'
calib_path = f'{analysis_run_prefix}_cross_signal_calibrator_summary.csv'
pred_export_path = f'{analysis_run_prefix}_cross_signal_decision_predictions.parquet'

metrics_df.to_csv(metrics_path, index=False)
tau_sweep_df.to_csv(tau_path, index=False)
domain_compare_df.to_csv(domain_path, index=False)
if not tau_at_budget_df.empty:
    tau_at_budget_df.to_csv(budget_path, index=False)
calibrator_df.to_csv(calib_path, index=False)
work_df.to_parquet(pred_export_path, index=False)

artifact_paths = {
    **bundle.artifact_paths,
    'cross_signal_metrics': metrics_path,
    'cross_signal_tau_sweep': tau_path,
    'cross_signal_domain_compare': domain_path,
    'cross_signal_calibrator_summary': calib_path,
    'cross_signal_decision_predictions': pred_export_path,
    **figure_paths,
}
if not tau_at_budget_df.empty:
    artifact_paths['cross_signal_tau_at_budget'] = budget_path

extra = {
    'analysis_run_prefix': str(analysis_run_prefix),
    'focus_label': str(FOCUS_LABEL),
    'risk_budget_tau': float(RISK_BUDGET_TAU),
    'bootstrap_samples': int(BOOTSTRAP_SAMPLES),
    'metric_rows': int(len(metrics_df)),
    'tau_rows': int(len(tau_sweep_df)),
    'inconclusive_tau_rows': int((tau_sweep_df['status'] == 'inconclusive').sum()) if not tau_sweep_df.empty else 0,
    'signal_count': int(metrics_df['signal'].nunique()) if not metrics_df.empty else 0,
    'figure_count': int(len(figure_paths)),
}

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name=NOTEBOOK_NAME,
    stage='cross_signal_decision_audit_completed',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='cross_signal_decision_audit_completed',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)

print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
print('exported_artifacts:')
for k, v in sorted(artifact_paths.items()):
    print('-', k, '->', v)


## Step 10 - Final Summary


In [ ]:
import numpy as np

nominal_df = metrics_df[metrics_df['domain'].astype(str).eq('nominal')].copy()
shifted_df = metrics_df[metrics_df['domain'].astype(str).eq('shifted')].copy()

best_nom = pd.DataFrame()
if not nominal_df.empty:
    best_nom = nominal_df.sort_values(['ece', 'nll', 'brier', 'auroc'], ascending=[True, True, True, False]).head(1)

print('=== Final Summary (Cross-Signal Decision Audit) ===')
print('focus_label =', FOCUS_LABEL, '| tau =', RISK_BUDGET_TAU)
print('signals_evaluated =', sorted(metrics_df['signal'].astype(str).unique().tolist()))
print('inconclusive_tau_rows =', int((tau_sweep_df['status'] == 'inconclusive').sum()) if not tau_sweep_df.empty else 0)

if not best_nom.empty:
    row = best_nom.iloc[0]
    print('[best nominal variant] ', f"{row['signal']}:{row['calibration']}")
    print('  AUROC=', f"{float(row['auroc']):.4f}",
          'AUPRC=', f"{float(row['auprc']):.4f}",
          'within_step_auc=', f"{float(row['within_step_auc']):.4f}",
          'ECE=', f"{float(row['ece']):.4f}",
          'NLL=', f"{float(row['nll']):.4f}",
          'Brier=', f"{float(row['brier']):.4f}")

if not tau_at_budget_df.empty:
    budget_view = tau_at_budget_df.copy()
    combo_view = budget_view[budget_view['signal'].astype(str).eq('combo')]
    if combo_view.empty:
        combo_view = budget_view
    cols = [
        'shift_suite', 'signal', 'calibration', 'tau', 'accept_rate',
        'false_safe', 'safe_reject', 'feasible_set_rate', 'fallback_rate', 'status', 'inconclusive_reason'
    ]
    cols = [c for c in cols if c in combo_view.columns]
    print('\\n[decision diagnostics near tau budget]')
    display(combo_view.loc[:, cols].head(30))

print('\\nInterpretation guideline:')
print('- If calibration lowers ECE/NLL/Brier and reduces false_safe at fixed tau, risk is more decision-grade.')
print('- If false_safe and safe_reject are both high, threshold rule and/or signal quality needs redesign.')
print('- Inconclusive rows indicate insufficient positive mass or accept/reject counts for stable claims.')
